In [0]:
# Live 4 (O) — Orquestração
# Notebook: 01_ingest_bronze_pedidos
# Objetivo: materializar Bronze em arquivos (CSV/JSON/PARQUET) no Volume, particionado por dt.

from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("tgt_schema", "aulas_ao_vivo.live_20260323")
dbutils.widgets.text("process_date", "2026-03-01")
dbutils.widgets.text("n_linhas", "500000")  # ASSUNÇÃO: default menor p/ demo; ajuste se quiser.
dbutils.widgets.text("seed", "66")
dbutils.widgets.text("start_ts", "2021-01-01 00:00:00")
dbutils.widgets.text("taxa_duplicidade", "0.08")

TGT_SCHEMA = dbutils.widgets.get("tgt_schema")
PROCESS_DATE = dbutils.widgets.get("process_date")
N_LINHAS = int(dbutils.widgets.get("n_linhas"))
SEED = int(dbutils.widgets.get("seed"))
START_TS = dbutils.widgets.get("start_ts")
TAXA_DUPLICIDADE = float(dbutils.widgets.get("taxa_duplicidade"))

VOLUME_PATH = f"/Volumes/{TGT_SCHEMA.replace('.', '/')}/files"
BASE_PATH = f"{VOLUME_PATH}/live_20260323"
BRONZE_BASE_PATH = f"{BASE_PATH}/bronze"

CSV_PATH = f"{BRONZE_BASE_PATH}/ecommerce/pedidos_csv/dt={PROCESS_DATE}"
JSON_PATH = f"{BRONZE_BASE_PATH}/ecommerce/pedidos_json/dt={PROCESS_DATE}"
PARQUET_PATH = f"{BRONZE_BASE_PATH}/ecommerce/pedidos_parquet/dt={PROCESS_DATE}"

def rm(path):
    try:
        dbutils.fs.rm(path, True)
    except Exception:
        pass

def generate_vendas_bronze(n_linhas: int, seed: int, start_ts: str, taxa_duplicidade: float = 0.0):
    if n_linhas < 1:
        return spark.createDataFrame([], schema=T.StructType([]))
    if taxa_duplicidade < 0 or taxa_duplicidade >= 1:
        raise ValueError("taxa_duplicidade deve estar entre 0 e 1 (exclusivo do 1).")

    ufs = ["AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"]
    produtos = ["Camiseta", "Tenis", "Caderno", "Mochila", "Fone", "Mouse", "Garrafa", "Jaqueta"]

    base = spark.range(n_linhas).withColumn("rnd", F.rand(seed))

    id_raw = F.concat(F.lit("PED"), F.lpad(F.col("id").cast("string"), 8, "0"))
    id_sujo = F.when(F.col("id") % 5 == 0, F.concat(F.lit(" "), id_raw, F.lit(" "))).otherwise(id_raw)

    start_unix = F.unix_timestamp(F.lit(start_ts), "yyyy-MM-dd HH:mm:ss")
    ts = F.to_timestamp(F.from_unixtime(start_unix + (F.col("id") * F.lit(3600))))
    fmt1 = F.date_format(ts, "yyyy-MM-dd HH:mm:ss")
    fmt2 = F.date_format(ts, "dd/MM/yyyy HH:mm")
    fmt3 = F.date_format(ts, "dd/MM/yyyy HH:mm:ss")
    data_suja = (F.when(F.col("id") % 3 == 0, fmt1).when(F.col("id") % 3 == 1, fmt2).otherwise(fmt3))

    idx_uf = ((F.col("id") % F.lit(len(ufs))) + F.lit(1)).cast("int")
    idx_prod = ((F.col("id") % F.lit(len(produtos))) + F.lit(1)).cast("int")

    uf_raw = F.element_at(F.array(*[F.lit(x) for x in ufs]), idx_uf)
    uf_sujo = (F.when(F.col("id") % 4 == 0, F.lower(uf_raw)).when(F.col("id") % 4 == 1, F.concat(F.lit(" "), uf_raw, F.lit(" "))).otherwise(uf_raw))

    prod_raw = F.element_at(F.array(*[F.lit(x) for x in produtos]), idx_prod)
    prod_sujo = (F.when(F.col("id") % 4 == 0, F.upper(prod_raw)).when(F.col("id") % 4 == 1, F.concat(F.lit(" "), prod_raw, F.lit(" "))).when(F.col("id") % 7 == 0, F.lit(" ")).otherwise(prod_raw))

    qty_num = (F.col("id") % F.lit(5)) + F.lit(1)
    qty_str = (F.when(F.col("id") % 4 == 0, F.lpad(qty_num.cast("string"), 2, "0")).when(F.col("id") % 9 == 0, F.lit(" ")).otherwise(qty_num.cast("string")))

    preco_base = ((F.col("id") % F.lit(20)) + F.lit(1)) * F.lit(10.5)
    valor_num = F.round(preco_base * qty_num.cast("double"), 2)
    valor_dot = F.format_string("%.2f", valor_num)
    valor_comma = F.regexp_replace(valor_dot, "\\.", ",")
    valor_sujo = (F.when(F.col("id") % 3 == 0, valor_dot).when(F.col("id") % 3 == 1, valor_comma).otherwise(F.concat(F.lit(" "), valor_comma, F.lit(" "))))

    status_sujo = (F.when(F.col("id") % 3 == 0, F.lit("Pago")).when(F.col("id") % 3 == 1, F.lit("CANCELADO")).otherwise(F.lit(" Pendente ")))

    df = base.select(
        id_sujo.alias("Id Pedido"),
        data_suja.alias("Data Pedido"),
        uf_sujo.alias("UF "),
        prod_sujo.alias("Produto"),
        qty_str.alias("Qtd"),
        valor_sujo.alias("Valor Total (R$)"),
        status_sujo.alias("Status")
    )

    if taxa_duplicidade > 0:
        duplicadas = df.sample(withReplacement=False, fraction=taxa_duplicidade, seed=seed + 999)
        df = df.unionByName(duplicadas)
    return df

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TGT_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {TGT_SCHEMA}.files")

rm(CSV_PATH); rm(JSON_PATH); rm(PARQUET_PATH)

vendas_bronze = generate_vendas_bronze(N_LINHAS, seed=SEED, start_ts=START_TS, taxa_duplicidade=TAXA_DUPLICIDADE)
print("Linhas bronze:", vendas_bronze.count())

bronze_write_df = vendas_bronze.repartition(8)
bronze_write_df.write.mode("overwrite").option("header", "true").csv(CSV_PATH)
bronze_write_df.write.mode("overwrite").json(JSON_PATH)
bronze_write_df.write.mode("overwrite").parquet(PARQUET_PATH)

print("CSV_PATH:", CSV_PATH)
print("JSON_PATH:", JSON_PATH)
print("PARQUET_PATH:", PARQUET_PATH)
